# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [5]:
from datetime import datetime
from agent import Agent

In [3]:
## TODO: Create the agent's instructions


ECOHOME_SYSTEM_PROMPT = """
You are the EcoHome Energy Advisor, an AI-powered assistant that helps households
optimize electricity usage to reduce cost and environmental impact.

Your responsibilities:
- Provide personalized, practical energy recommendations for homes with
  solar panels, electric vehicles, smart thermostats, and appliances.
- Optimize for BOTH electricity cost savings and carbon footprint reduction.
- Base your advice on available data such as:
  - Weather forecasts and solar irradiance
  - Time-of-use electricity pricing
  - Historical energy usage and solar generation
  - Proven energy-saving best practices from retrieved documents

How you should operate:
- Use tools whenever relevant to retrieve weather, pricing, usage history,
  solar generation data, or energy-saving tips.
- Combine insights from multiple tools before making recommendations.
- When suggesting optimizations (e.g., changing run time, charging schedule),
  explain the reasoning clearly and quantify potential savings when possible.
- Prefer energy usage during high solar generation and low-price periods.
- Be concise, clear, and actionable. Avoid generic advice.

Response style:
- Start with a direct answer to the user’s question.
- Explain WHY the recommendation is optimal (pricing, solar, history, or efficiency).
- Include estimated cost or energy savings if applicable.
- Use a helpful, professional, and friendly tone.
- If data is limited or assumptions are made, state them clearly.

Constraints:
- Do not invent data. Rely on tool outputs and reasonable assumptions.
- Do not mention internal implementation details, code, or tool names explicitly.
- If a question cannot be answered with available data, explain what information
  is missing and provide best-effort guidance.

Your goal is to help users save money, use cleaner energy, and make smarter
day-to-day energy decisions.
"""



In [6]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [7]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [8]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow, I recommend charging between **10 AM and 2 PM**.

### Here's why this is optimal:

1. **Solar Generation**: The solar generation data indicates that there is no solar generation expected tomorrow, which means charging during the day won't utilize solar energy. However, charging during the day can still be beneficial if you can take advantage of lower electricity rates.

2. **Electricity Pricing**: The time-of-use pricing shows that the rates are lowest during the early morning and late evening:
   - **Off-Peak Rates**: 
     - 12 AM - 6 AM: Rates as low as $0.08 to $0.09 per kWh.
     - 10 PM - 12 AM: Rates drop to $0.079 to $0.084 per kWh.
   - **Mid-Peak Rates**: 
     - 10 AM - 2 PM: Rates are around $0.116 to $0.137 per kWh.

3. **Recommendation**: 
   - **Charge Overnight**: If you can charge your EV overnight (from 12 AM to 6 AM), you will benefit from the lowest rates. For example, char

In [9]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices
- query_solar_generation


## 2. Define Test Cases

In [11]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [13]:

test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should recommend optimal charging time considering solar availability and electricity prices."
    },
    {
        "id": "ev_charging_2",
        "question": "Should I charge my EV during the day or at night if I want the lowest electricity cost?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should compare day vs night rates and suggest the cheaper option."
    },
    {
        "id": "thermostat_1",
        "question": "What thermostat settings should I use tomorrow to reduce cooling costs?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should suggest temperature settings based on weather conditions to reduce HVAC usage."
    },
    {
        "id": "thermostat_2",
        "question": "My AC is running a lot. How can I reduce HVAC energy consumption?",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "The response should identify HVAC usage patterns and recommend energy-saving actions."
    },
    {
        "id": "appliance_1",
        "question": "When is the best time today to run my dishwasher to save money?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should recommend running the appliance during off-peak or low-cost hours."
    },
    {
        "id": "appliance_2",
        "question": "Should I run my washing machine now or later to save electricity cost?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "The response should compare current and later electricity rates and recommend the optimal time."
    },
    {
        "id": "solar_1",
        "question": "How can I maximize the use of my solar power today?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation"],
        "expected_response": "The response should suggest shifting usage to high solar generation hours."
    },
    {
        "id": "solar_2",
        "question": "Is today a good day to rely more on solar energy for household usage?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "The response should assess solar potential based on weather conditions."
    },
    {
        "id": "cost_savings_1",
        "question": "How much money can I save by shifting EV charging from peak to off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should estimate cost savings and explain the calculation assumptions."
    },
    {
        "id": "summary_1",
        "question": "Can you summarize my energy usage over the last 24 hours?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should provide a concise summary of consumption, cost, and major contributing devices."
    }
]


if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [14]:
CONTEXT = "Location: San Francisco, CA"

In [15]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: Should I charge my EV during the day or at night if I want the lowest electricity cost?
--------------------------------------------------

Test 3: thermostat_1
Question: What thermostat settings should I use tomorrow to reduce cooling costs?
--------------------------------------------------

Test 4: thermostat_2
Question: My AC is running a lot. How can I reduce HVAC energy consumption?
--------------------------------------------------

Test 5: appliance_1
Question: When is the best time today to run my dishwasher to save money?
--------------------------------------------------

Test 6: appliance_2
Question: Should I run my washing machine now or later to save electricity cost?
--------------------------------------------------

Test 7: solar_1
Questio

In [16]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='ff0a2d90-8a01-4856-abfb-237e793597a6'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='9001e45a-deff-490f-a32f-7d114c4fa353'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Ca870SEl7bljm43cmkbGmmtJ', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 1}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_e1gJYkBS0GNfQ8mbdB49VJQE', 'function': {'arguments': '{"date": "2023-10-07"}', 'name': 'get_electricity_prices'}, 'type': 'function'}, {'id': 'call_OxsQ12LH9snD3kmQlpHX5ehL', 'function': {'arguments': '{"start_date": "2023-10-07", "end_date": "2023-

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [17]:
# TODO: Create a response evaluator

def evaluate_response(question, final_response, expected_response):
    """
    Evaluate a single response against the expected response.

    Evaluation logic:
    - Checks that a meaningful response exists
    - Ensures the response is relevant to the question
    - Verifies that expected concepts are covered at a high level

    Returns:
        dict with pass/fail status and explanation
    """
    if not final_response or not isinstance(final_response, str):
        return {
            "passed": False,
            "reason": "Response is empty or not a valid string."
        }

    # Normalize text for simple semantic checks
    response_text = final_response.lower()
    expected_text = expected_response.lower()

    # Basic relevance check: response should mention at least one key concept
    # derived from the expected response description
    expected_keywords = [
        word for word in expected_text.split()
        if len(word) > 4  # ignore very common short words
    ]

    matched_keywords = [
        word for word in expected_keywords
        if word in response_text
    ]

    if len(matched_keywords) == 0:
        return {
            "passed": False,
            "reason": (
                "Response does not appear to address the expected concepts. "
                "It may be too generic or off-topic."
            )
        }

    return {
        "passed": True,
        "reason": (
            "Response is relevant, addresses the question, "
            "and covers expected concepts at a high level."
        )
    }


In [ ]:

final_response = result.final_output  # output from agent run

response_eval = evaluate_response(
    question=test["question"],
    final_response=final_response,
    expected_response=test["expected_response"]
)

print(response_eval)


In [18]:
# TODO: Create a tool udage evaluator

def evaluate_tool_usage(messages, expected_tools):
    """
    Evaluate whether the expected tools were used by the agent.

    Args:
        messages (list): List of messages exchanged during agent execution
        expected_tools (list): List of tool names expected to be used

    Returns:
        dict: Evaluation result with pass/fail and explanation
    """
    if not messages or not expected_tools:
        return {
            "passed": False,
            "reason": "No messages or expected tools provided for evaluation."
        }

    # Extract tool names used from tool messages
    used_tools = set()

    for msg in messages:
        # Tool messages usually have a 'name' attribute
        tool_name = getattr(msg, "name", None)
        if tool_name:
            used_tools.add(tool_name)

    if not used_tools:
        return {
            "passed": False,
            "reason": "No tools appear to have been used in this interaction."
        }

    # Check overlap between expected and used tools
    matched_tools = set(expected_tools).intersection(used_tools)

    if len(matched_tools) == 0:
        return {
            "passed": False,
            "reason": (
                f"Expected tools {expected_tools}, but none were used. "
                f"Tools actually used: {list(used_tools)}"
            )
        }

    return {
        "passed": True,
        "reason": (
            f"Tool usage is appropriate. Expected tools matched: {list(matched_tools)}. "
            f"Tools used: {list(used_tools)}"
        )
    }


In [19]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement

def generate_evaluation_report(evaluation_results):
    """
    Generate a comprehensive evaluation report for the EcoHome Energy Advisor.

    Args:
        evaluation_results (list): List of dicts containing per-test evaluation results.
                                   Each item is expected to have:
                                   - test_id
                                   - response_evaluation {passed, reason}
                                   - tool_evaluation {passed, reason}

    Returns:
        dict: Consolidated evaluation report
    """
    if not evaluation_results:
        raise ValueError("No evaluation results provided.")

    total_tests = len(evaluation_results)
    response_pass = 0
    tool_pass = 0

    failed_response_cases = []
    failed_tool_cases = []

    for result in evaluation_results:
        if result["response_evaluation"]["passed"]:
            response_pass += 1
        else:
            failed_response_cases.append({
                "id": result["test_id"],
                "reason": result["response_evaluation"]["reason"]
            })

        if result["tool_evaluation"]["passed"]:
            tool_pass += 1
        else:
            failed_tool_cases.append({
                "id": result["test_id"],
                "reason": result["tool_evaluation"]["reason"]
            })

    response_score = round((response_pass / total_tests) * 100, 1)
    tool_score = round((tool_pass / total_tests) * 100, 1)
    overall_score = round((response_score + tool_score) / 2, 1)

    # Strengths & weaknesses
    strengths = []
    weaknesses = []
    recommendations = []

    if response_score >= 80:
        strengths.append("High-quality, context-aware responses with clear recommendations.")
    else:
        weaknesses.append("Some responses lack sufficient relevance or actionable detail.")
        recommendations.append(
            "Improve response clarity by explicitly linking recommendations to cost, solar, or usage data."
        )

    if tool_score >= 80:
        strengths.append("Appropriate and consistent use of tools to support reasoning.")
    else:
        weaknesses.append("Expected tools were missed in some scenarios.")
        recommendations.append(
            "Encourage multi-tool reasoning in prompts, especially combining pricing and weather insights."
        )

    if not strengths:
        strengths.append("Core agent structure and tooling are correctly implemented.")

    report = {
        "summary": {
            "total_tests": total_tests,
            "response_quality_score_percent": response_score,
            "tool_usage_score_percent": tool_score,
            "overall_score_percent": overall_score
        },
        "strengths": strengths,
        "weaknesses": weaknesses,
        "failed_response_cases": failed_response_cases,
        "failed_tool_cases": failed_tool_cases,
        "recommendations": recommendations or [
            "Current performance is strong. Focus on expanding scenario coverage and edge-case handling."
        ]
    }

    # Pretty print report (important for notebooks / reviewers)
    print("\n" + "=" * 80)
    print("ECOHOME ENERGY ADVISOR – EVALUATION REPORT")
    print("=" * 80)
    print(f"Total Test Cases               : {total_tests}")
    print(f"Response Quality Score (%)     : {response_score}")
    print(f"Tool Usage Score (%)           : {tool_score}")
    print(f"OVERALL SCORE (%)              : {overall_score}")
    print("\nStrengths:")
    for s in report["strengths"]:
        print(f"  - {s}")

    if weaknesses:
        print("\nWeaknesses:")
        for w in report["weaknesses"]:
            print(f"  - {w}")

    if recommendations:
        print("\nRecommendations:")
        for r in report["recommendations"]:
            print(f"  - {r}")

    print("\nDetailed Failures:")
    if failed_response_cases:
        print("  Response Issues:")
        for f in failed_response_cases:
            print(f"    - {f['id']}: {f['reason']}")
    else:
        print("  Response Issues: None")

    if failed_tool_cases:
        print("  Tool Usage Issues:")
        for f in failed_tool_cases:
            print(f"    - {f['id']}: {f['reason']}")
    else:
        print("  Tool Usage Issues: None")

    print("=" * 80 + "\n")

    return report


In [24]:
print(test_results[0].keys())

dict_keys(['test_id', 'question', 'response', 'expected_tools', 'expected_response', 'timestamp'])


In [25]:


evaluation_results = []

for result in test_results:

    final_response = result["response"]          
    question = result["question"]
    expected_response = result["expected_response"]
    expected_tools = result["expected_tools"]

    # --- Evaluate final response ---
    response_eval = evaluate_response(
        question=question,
        final_response=final_response,
        expected_response=expected_response
    )

    # --- Evaluate tool usage ---
    tool_eval = evaluate_tool_usage(
        messages=result.get("messages", []),   
        expected_tools=expected_tools
    )

    evaluation_results.append({
        "test_id": result["test_id"],
        "response_evaluation": response_eval,
        "tool_evaluation": tool_eval
    })



In [26]:
print(evaluation_results)

[{'test_id': 'ev_charging_1', 'response_evaluation': {'passed': False, 'reason': 'Response is empty or not a valid string.'}, 'tool_evaluation': {'passed': False, 'reason': 'No messages or expected tools provided for evaluation.'}}, {'test_id': 'ev_charging_2', 'response_evaluation': {'passed': False, 'reason': 'Response is empty or not a valid string.'}, 'tool_evaluation': {'passed': False, 'reason': 'No messages or expected tools provided for evaluation.'}}, {'test_id': 'thermostat_1', 'response_evaluation': {'passed': False, 'reason': 'Response is empty or not a valid string.'}, 'tool_evaluation': {'passed': False, 'reason': 'No messages or expected tools provided for evaluation.'}}, {'test_id': 'thermostat_2', 'response_evaluation': {'passed': False, 'reason': 'Response is empty or not a valid string.'}, 'tool_evaluation': {'passed': False, 'reason': 'No messages or expected tools provided for evaluation.'}}, {'test_id': 'appliance_1', 'response_evaluation': {'passed': False, 'reas